# Optimize Boron concentration for a target K_eff using OpenMC tally derivatives: An alternative to search_for_keff (openmc python API).

This notebook demonstrates a gradient-based approach using OpenMC tally derivatives to find a critical boron concentration (ppm) and compares it with the built-in `openmc.search_for_keff` method. Cells below break the original script into smaller pieces with explanatory text and add experiments to test sensitivity to numerical and simulation parameters.

In [5]:
#!/usr/bin/env python3
"""
gradient_optimization_demo.py - Demonstrating gradient-based optimization speedup with plotting
"""

# Core imports and configuration
import os
import math
import h5py
import openmc
import numpy as np
import warnings
import matplotlib.pyplot as plt

# Suppress FutureWarnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)

# Physical/constants used in chain-rule conversions
N_A = 6.02214076e23     # Avogadro's number (atoms/mol)
A_B_nat = 10.81         # g/mol approximate atomic mass for natural boron

#os.environ['PATH'] = '/workspaces/openmc/build/bin/' + os.environ['PATH']
#os.environ['OPENMC_CROSS_SECTIONS'] = '/home/codespace/nndc_hdf5/cross_sections.xml'
print("PATH:", os.environ.get('PATH'))
print("OPENMC_CROSS_SECTIONS:", os.environ.get('OPENMC_CROSS_SECTIONS'))
os.environ['PATH'] = '/workspaces/openmc/build/bin/:' + os.environ['PATH']
os.environ['OPENMC_CROSS_SECTIONS'] = '/home/codespace/nndc_hdf5/cross_sections.xml'


PATH: /home/codespace/.python/current/bin:/vscode/bin/linux-x64/bf9252a2fb45be6893dd8870c0bf37e2e1766d61/bin/remote-cli:/home/codespace/.local/bin:/home/codespace/.dotnet:/home/codespace/nvm/current/bin:/home/codespace/.php/current/bin:/home/codespace/.python/current/bin:/home/codespace/java/current/bin:/home/codespace/.ruby/current/bin:/home/codespace/.local/bin:/usr/local/python/current/bin:/usr/local/py-utils/bin:/usr/local/jupyter:/usr/local/oryx:/usr/local/go/bin:/go/bin:/usr/local/sdkman/bin:/usr/local/sdkman/candidates/java/current/bin:/usr/local/sdkman/candidates/gradle/current/bin:/usr/local/sdkman/candidates/maven/current/bin:/usr/local/sdkman/candidates/ant/current/bin:/usr/local/rvm/gems/default/bin:/usr/local/rvm/gems/default@global/bin:/usr/local/rvm/rubies/default/bin:/usr/local/share/rbenv/bin:/usr/local/php/current/bin:/opt/conda/bin:/usr/local/nvs:/usr/local/share/nvm/current/bin:/usr/local/hugo/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/sha

**Model builder**: This cell contains the `build_model(ppm_Boron)` function that constructs materials, geometry, and settings for the simple pin-cell model used in the experiments. Keep this function as-is to ensure reproducibility.

In [6]:
print("PATH:", os.environ.get('PATH'))
print("OPENMC_CROSS_SECTIONS:", os.environ.get('OPENMC_CROSS_SECTIONS'))

# ===============================================================
# Model builder
# ===============================================================
def build_model(ppm_Boron):
    # Create the pin materials
    fuel = openmc.Material(name='1.6% Fuel', material_id=1)
    fuel.set_density('g/cm3', 10.31341)
    fuel.add_element('U', 1., enrichment=1.6)
    fuel.add_element('O', 2.)

    zircaloy = openmc.Material(name='Zircaloy', material_id=2)
    zircaloy.set_density('g/cm3', 6.55)
    zircaloy.add_element('Zr', 1.)

    water = openmc.Material(name='Borated Water', material_id=3)
    water.set_density('g/cm3', 0.741)
    water.add_element('H', 2.)
    water.add_element('O', 1.)
    water.add_element('B', ppm_Boron * 1e-6)

    materials = openmc.Materials([fuel, zircaloy, water])

    # Geometry
    fuel_outer_radius = openmc.ZCylinder(r=0.39218)
    clad_outer_radius = openmc.ZCylinder(r=0.45720)

    min_x = openmc.XPlane(x0=-0.63, boundary_type='reflective')
    max_x = openmc.XPlane(x0=+0.63, boundary_type='reflective')
    min_y = openmc.YPlane(y0=-0.63, boundary_type='reflective')
    max_y = openmc.YPlane(y0=+0.63, boundary_type='reflective')

    fuel_cell = openmc.Cell(name='1.6% Fuel')
    fuel_cell.fill = fuel
    fuel_cell.region = -fuel_outer_radius

    clad_cell = openmc.Cell(name='1.6% Clad')
    clad_cell.fill = zircaloy
    clad_cell.region = +fuel_outer_radius & -clad_outer_radius

    moderator_cell = openmc.Cell(name='1.6% Moderator')
    moderator_cell.fill = water
    moderator_cell.region = +clad_outer_radius & (+min_x & -max_x & +min_y & -max_y)

    root_universe = openmc.Universe(name='root universe', universe_id=0)
    root_universe.add_cells([fuel_cell, clad_cell, moderator_cell])

    geometry = openmc.Geometry(root_universe)

    # Settings
    settings = openmc.Settings()
    settings.batches = 300
    settings.inactive = 20
    settings.particles = 1000
    settings.run_mode = 'eigenvalue'
    settings.verbosity=1

    bounds = [-0.63, -0.63, -10, 0.63, 0.63, 10.]
    uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:], only_fissionable=True)
    settings.source = openmc.Source(space=uniform_dist)

    model = openmc.model.Model(geometry, materials, settings)
    return model

PATH: /workspaces/openmc/build/bin/:/home/codespace/.python/current/bin:/vscode/bin/linux-x64/bf9252a2fb45be6893dd8870c0bf37e2e1766d61/bin/remote-cli:/home/codespace/.local/bin:/home/codespace/.dotnet:/home/codespace/nvm/current/bin:/home/codespace/.php/current/bin:/home/codespace/.python/current/bin:/home/codespace/java/current/bin:/home/codespace/.ruby/current/bin:/home/codespace/.local/bin:/usr/local/python/current/bin:/usr/local/py-utils/bin:/usr/local/jupyter:/usr/local/oryx:/usr/local/go/bin:/go/bin:/usr/local/sdkman/bin:/usr/local/sdkman/candidates/java/current/bin:/usr/local/sdkman/candidates/gradle/current/bin:/usr/local/sdkman/candidates/maven/current/bin:/usr/local/sdkman/candidates/ant/current/bin:/usr/local/rvm/gems/default/bin:/usr/local/rvm/gems/default@global/bin:/usr/local/rvm/rubies/default/bin:/usr/local/share/rbenv/bin:/usr/local/php/current/bin:/opt/conda/bin:/usr/local/nvs:/usr/local/share/nvm/current/bin:/usr/local/hugo/bin:/usr/local/sbin:/usr/local/bin:/usr/sbi

**Helpers**: utility functions for extracting cell IDs used by a material and for running OpenMC with derivative tallies.

In [12]:
# helper to get the material id
def find_cells_using_material(geometry, material):
    "Return list of cell ids in `geometry` filled with `material`."
    return [c.id for c in geometry.get_all_cells().values() if c.fill is material]


**Running OpenMC with derivative tallies**: `run_with_gradient` runs an OpenMC simulation for a given boron ppm, attaches derivative tallies for the boron isotopes, and returns k-eff plus the required tallies to compute dk/dppm. Keep this implementation intact so gradient calculations remain consistent with the original notebook.

In [8]:
def run_with_gradient(ppm_B, target_batches=50, water_material_id=3, boron_nuclides=('B10', 'B11')):
    """Run OpenMC and compute k-effective with gradient information"""
    # Clean up previous files
    for f in ['summary.h5', f'statepoint.{target_batches}.h5', 'tallies.out']:
        if os.path.exists(f):
            os.remove(f)

    # Build model
    model = build_model(ppm_B)

    # Auto-detect moderator cells
    water = model.materials[water_material_id - 1]  # Materials are 0-indexed
    moderator_cell_ids = find_cells_using_material(model.geometry, water)
    moderator_filter = openmc.CellFilter(moderator_cell_ids)

    # Base tallies
    tF_base = openmc.Tally(name='FissionBase')
    tF_base.scores = ['nu-fission']
    tA_base = openmc.Tally(name='AbsorptionBase')
    tA_base.scores = ['absorption']

    # Derivative tallies
    deriv_tallies = []
    for nuc in boron_nuclides:
        deriv = openmc.TallyDerivative(
            variable='nuclide_density',
            material=water_material_id,
            nuclide=nuc
        )

        tf = openmc.Tally(name=f'Fission_deriv_{nuc}')
        tf.scores = ['nu-fission']
        tf.derivative = deriv
        tf.filters = [moderator_filter]

        ta = openmc.Tally(name=f'Absorp_deriv_{nuc}')
        ta.scores = ['absorption']
        ta.derivative = deriv
        ta.filters = [moderator_filter]

        deriv_tallies += [tf, ta]

    model.tallies = openmc.Tallies([tF_base, tA_base] + deriv_tallies)
    model.settings.batches = target_batches
    model.settings.inactive = max(1, int(target_batches * 0.1))

    # Run simulation
    model.run()
    sp = openmc.StatePoint(f"statepoint.{target_batches}.h5")

    # Get results
    k_eff = sp.keff.nominal_value

    # Base tallies
    fission_tally = sp.get_tally(name='FissionBase')
    absorption_tally = sp.get_tally(name='AbsorptionBase')
    F_base = float(np.sum(fission_tally.mean))
    A_base = float(np.sum(absorption_tally.mean))

    # Derivative tallies
    dF_dN_total = 0.0
    dA_dN_total = 0.0

    for nuc in boron_nuclides:
        fission_deriv = sp.get_tally(name=f'Fission_deriv_{nuc}')
        absorption_deriv = sp.get_tally(name=f'Absorp_deriv_{nuc}')

        if fission_deriv:
            dF_dN_total += float(np.sum(fission_deriv.mean))
        if absorption_deriv:
            dA_dN_total += float(np.sum(absorption_deriv.mean))

    return k_eff, F_base, A_base, dF_dN_total, dA_dN_total, water.density

**Gradient-based optimizer**: the gradient descent routine that uses the analytical derivative tallies to propose ppm updates. The original adaptive logic is preserved; we add an experiments cell later to vary tuning parameters.

### Theory, derivation and scaling

This optimizer uses derivative tallies to estimate how small changes in boron concentration (ppm) affect the reactor multiplication factor k_eff. The notebook treats k approximately as the ratio of a fission production tally F to an absorption tally A, i.e. k ≈ F / A. Both F and A depend on the boron number density N (atoms/cm³).

From calculus (quotient rule) we get the derivative of k with respect to N:

- dk/dN = (A * dF_dN - F * dA_dN) / A^2

This expression follows directly from d(f/g)/dN = (g f' - f g') / g^2 (standard calculus — see any calculus text).

To convert from mass-part-per-million (ppm) to number density we use the mass fraction and Avogadro's number. If ppm is a mass-part-per-million, the boron mass fraction is ppm × 1e-6 and the boron number density is

- N = (rho_water * mass_fraction) × (N_A / A_B)  
so
- dN/dppm = 1e-6 × rho_water × N_A / A_B

Finally, combine with the chain rule: dk/dppm = dk/dN × dN/dppm.

### Short numeric scaling sanity check (why derivatives appear extremely large)

Using representative constants from the notebook:
- N_A ≈ 6.022×10^23 mol⁻¹ (Avogadro)
- A_B_nat ≈ 10.8 g/mol
- rho_water ≈ 0.741 g/cm³ (model value)

Then

- dN/dppm ≈ 1e-6 × 0.741 × (6.022e23 / 10.8) ≈ 4×10^16 atoms·cm⁻3 per ppm

So dk/dppm = dk/dN × (≈4×10^16). If dk/dN is O(10^4) (which depends on your absolute tally magnitudes), dk/dppm can be O(10^20). This is why per‑ppm derivatives look enormous — the ppm→atoms conversion multiplies by Avogadro-scale factors.

### Practical considerations and quick recommendations

- Large dk/dppm magnitudes are expected numerically because ppm is a very small mass fraction but corresponds to a large change in atom counts when converted to atoms/cm³. Expect amplification by ~1e16–1e17 from the conversion alone.
- When using these derivatives in an optimizer you should scale or clip updates to keep steps physically reasonable (e.g., clamp per-iteration ppm changes, perform line search/backtracking, optimize over log(ppm) rather than linear ppm, or normalize the gradient to a target step magnitude).
- Also account for stochastic noise: derivative tallies from Monte Carlo will have statistical uncertainty. Increase batches/particles or use averaging/adjoint methods for more robust gradients.

### References

- Stewart, J. — "Calculus" (quotient/chain rules) — standard calculus texts for the quotient and chain rules.
- Lamarsh, J. R.; Baratta, A. J. — "Introduction to Nuclear Reactor Theory" (perturbation and sensitivity of k-effective)
- Duderstadt, J. J.; Hamilton, L. J. — "Nuclear Reactor Analysis" (first-order perturbation results)
- Bell, G.; Glasstone, S. — "Nuclear Reactor Theory"
- Lewis, E. E.; Miller, W. F. Jr. — "Computational Methods of Neutron Transport" (adjoint/perturbation methods)
- OpenMC documentation — tally derivatives/TallyDerivative (practical implementation and usage guidance)

These citations justify the quotient/chain rule usage and the expected scaling when converting ppm → atoms/cm³.

In [1]:
def gradient_based_search(ppm_start, ppm_range, k_target, max_iter, tol=1e-2, initial_learning_rate=1e-17):
    """Gradient-based optimization using analytical derivatives with adaptive learning rate"""
    ppm = float(ppm_start)
    history = []

    # Adaptive learning rate parameters
    learning_rate = initial_learning_rate
    lr_increase_factor = 1.5  # Increase LR when making good progress
    lr_decrease_factor = 0.5  # Decrease LR when oscillating or diverging
    max_learning_rate = 1e-16
    min_learning_rate = 1e-18

    # For tracking progress
    prev_error = None
    consecutive_improvements = 0
    consecutive_worsening = 0

    print("GRADIENT-BASED OPTIMIZATION WITH ADAPTIVE LEARNING RATE")
    print(f"Initial: {ppm:.1f} ppm, Target: k = {k_target}")
    print("Iter |   ppm   |   k_eff   |  Error  |  Gradient  |  Step  | Learning Rate")
    print("-" * 85)

    for it in range(max_iter):
        k, F, A, dF_dN, dA_dN, rho_water = run_with_gradient(ppm)
        err = k - k_target
        history.append((ppm, k, err, dF_dN, dA_dN, learning_rate))

        # Calculate gradient using chain rule
        dk_dN = (A * dF_dN - F * dA_dN) / (A * A)
        dN_dppm = 1e-6 * rho_water * N_A / A_B_nat
        dk_dppm = dk_dN * dN_dppm
        #dk_dppm = dk_dN # assuming dN_dppm is approximately 1 for simplicity

        prev_error = err

        # Gradient descent step with momentum-like behavior for small gradients
        if abs(dk_dppm) < 1e-10:  # Very small gradient
            # Use a conservative fixed step in the right direction
            step = -100 if err > 0 else 100
        else:
            step = -learning_rate * err * dk_dppm

        # Additional adaptive scaling based on error magnitude
        error_magnitude = abs(err)
        if error_magnitude > 0.1:
            step *= 1.5
        elif error_magnitude < 0.01:
            step *= 0.7

        ppm_new = ppm + step
        print("NEW PPM SUGGESTION: ", ppm_new)
        print("ERROR WAS: ", err)        
        # Apply bounds
        ppm_new = max(ppm_range[0], min(ppm_new, ppm_range[1]))


        print(f"{it+1:3d} | {ppm:7.1f} | {k:9.6f} | {err:7.4f} | {dk_dppm:10.2e} | {step:7.1f} | {learning_rate:12.2e}")

        if abs(err) < tol:
            print(f"✓ CONVERGED in {it+1} iterations")
            return ppm, history

        ppm = ppm_new

    print(f"Reached maximum iterations ({max_iter})")
    return ppm, history

**OpenMC built-in keff search**: wrapper around `openmc.search_for_keff` used for baseline comparisons.

In [2]:
def builtin_keff_search(k_target, ppm_start, ppm_range, max_iter):
    """Call `openmc.search_for_keff` with a small bracket and return results."""
    print("\n===== OPENMC BUILTIN KEFF SEARCH =====\n")

    crit_ppm, guesses, keffs = openmc.search_for_keff(
        build_model,
        initial_guess = ppm_start,
        target=k_target,
        bracket=ppm_range,
        tol=1e-2,
        print_iterations=True,
        run_args={'output': False}
    )

    print("\nCritical Boron Concentration: {:4.0f} ppm".format(crit_ppm))
    return crit_ppm, guesses, keffs

**Comparison function**: run both methods and summarize results. This function keeps the previous comparison logic intact.

In [10]:
def compare_optimization_methods(ppm_start, k_target, ppm_range, max_iter):
    """Compare gradient-based vs built-in search and summarize results."""
    print("=" * 80)
    print("COMPARING OPTIMIZATION METHODS FOR BORON CONCENTRATION SEARCH")
    print(f"Target k_eff: {k_target}, Initial guess: {ppm_start} ppm")
    print("=" * 80)
    '''
    # Method 1: OpenMC function (gradient-free)
    print("\n=== Running OpenMC built-in keff search ===")
    builtin_ppm, guesses, keffs = builtin_keff_search(k_target, ppm_start, ppm_range, max_iter)
    # Convert built-in search logs to unified history format (approximate)
    builtin_history = [(g, k, k - 1.0, 0, 0) for g, k in zip(guesses, keffs)]
    '''
    # Method 2: Gradient-based (analytical derivatives)
    grad_ppm, grad_history = gradient_based_search(ppm_start, ppm_range, k_target, max_iter)

    methods = [
        ("Analytical Gradient", grad_ppm, grad_history),
        ("OpenMC Built-in", builtin_ppm, builtin_history),
    ]

    # Results comparison
    print("\n" + "=" * 80)
    print("FINAL RESULTS COMPARISON")
    print("=" * 80)

    best_method = None
    best_error = float('inf')

    for name, ppm, history in methods:
        if history:
            final_k = history[-1][1]
            final_err = abs(history[-1][2])
            iterations = len(history)
            print(f"\n{name}:")
            print(f"  Final ppm: {ppm:.1f}")
            print(f"  Final k_eff: {final_k:.6f}")
            print(f"  Final error: {final_err:.6f}")
            print(f"  Iterations: {iterations}")
            if final_err < best_error:
                best_error = final_err
                best_method = name

    if best_method:
        print(f"\n★ BEST METHOD: {best_method} (error = {best_error:.6f})")

    # Convergence speed analysis
    print(f"\nCONVERGENCE SPEED ANALYSIS:")
    tolerance_levels = [0.05, 0.02, 0.01]  # 5%, 2%, 1% tolerance
    for name, ppm, history in methods:
        if history:
            print(f"\n{name}:")
            for tol_level in tolerance_levels:
                iterations_to_tolerance = None
                for i, (_, k, err, *_) in enumerate(history):
                    if abs(err) < tol_level:
                        iterations_to_tolerance = i + 1
                        break
                if iterations_to_tolerance:
                    print(f"  Reached {tol_level*100:.0f}% tolerance in {iterations_to_tolerance} iterations")
                else:
                    print(f"  Did not reach {tol_level*100:.0f}% tolerance")

    return methods

**Experiment cells (1/N)**: 

1. run sensitivity tests to evaluate how simulation settings and optimizer hyperparameters affect reliability. The experiments below are intentionally conservative (use few batches/particles) so they can be executed quickly as smoke tests; increase the counts for production runs.

In [8]:
def run_experiments():
    """Run small experiments that vary: batches, particles, initial_learning_rate, and initial ppm."""
    experiments = []

    # Example parameter sweep (kept small for quick runs)
    sweeps = {
        'batches': [30, 50],
        'particles': [200, 500],
        'initial_lr': [1e-16, 1e-18],
        'ppm_start': [800.0, 1200.0],
    }

    for b in sweeps['batches']:
        for p in sweeps['particles']:
            for lr in sweeps['initial_lr']:
                for ppm0 in sweeps['ppm_start']:
                    # Adjust settings for a quick smoke-run
                    openmc.settings = None  # ensure global state not reused
                    # Update build_model default settings by constructing a custom model inside run_with_gradient via monkeypatching batches/particles is non-trivial here,
                    # so we simply call run_with_gradient with target_batches=b; the model uses settings from build_model except batches overwritten in run_with_gradient.
                    try:
                        k, F, A, dF_dN, dA_dN, rho = run_with_gradient(ppm0, target_batches=b)
                    except Exception as e:
                        print('Run failed (this is expected for short/quick settings):', e)
                        k = None
                    experiments.append({'batches': b, 'particles': p, 'initial_lr': lr, 'ppm0': ppm0, 'k': k})
    return experiments

# A small helper to plot a provided history from gradient-based runs
def plot_history(history, title='Optimization history'):
    if not history:
        print('No history to plot')
        return
    pvals = [h[0] for h in history]
    keffs = [h[1] for h in history]
    errs = [h[2] for h in history]
    fig, ax = plt.subplots(1,2, figsize=(12,4))
    ax[0].plot(pvals, marker='o')
    ax[0].set_title('ppm over iterations')
    ax[0].set_xlabel('iteration')
    ax[1].plot(keffs, marker='o')
    ax[1].set_title('k_eff over iterations')
    ax[1].axhline(1.0, color='k', linestyle='--')
    plt.suptitle(title)
    plt.show()

## **Experiment cells (2/N)**: 
1. Test convergence across subcritical, supercritical and critical regimes for both approaches (**TODO**)

**One-shot ppm update (using derivative tallies)**: compute dk/dppm from the derivative tallies produced by `run_with_gradient` and propose a single Newton-like ppm update intended to reach a target `k_eff`. This cell runs a small-batch smoke-test by default; increase `target_batches` for production runs.

In [11]:
def one_shot_ppm_update(ppm_B, k_target=1.0, target_batches=30, boron_nuclides=('B10','B11')):
    """Use `run_with_gradient` to estimate dk/dppm and recommend a one-shot ppm update."""
    # Run the existing helper which attaches derivative tallies and returns the pieces we need
    k, F, A, dF_dN, dA_dN, rho = run_with_gradient(ppm_B, target_batches=target_batches, boron_nuclides=boron_nuclides)

    if A == 0.0:
        raise RuntimeError('Absorption base tally is zero; cannot compute dk/dN')

    # Chain rule: dk/dN and dk/dppm
    dk_dN = (A * dF_dN - F * dA_dN) / (A * A)
    # Physical constants (defined earlier in the notebook): N_A, A_B_nat
    dN_dppm = 1e-6 * rho * N_A / A_B_nat
    dk_dppm = dk_dN * dN_dppm

    result = {'ppm': ppm_B, 'k': k, 'dk_dppm': dk_dppm}

    if abs(dk_dppm) < 1e-20:
        result['recommended_ppm'] = None
        result['note'] = 'Gradient too small to recommend an update'
    else:
        recommended_ppm = ppm_B + (k_target - k) / dk_dppm
        result['recommended_ppm'] = recommended_ppm

    return result



# Quick smoke-run example (small batches) -- adjust `target_batches` for more reliable estimates.
res_one_shot = one_shot_ppm_update(1000.0, k_target=0.85, target_batches=300)
print('One-shot recommendation (quick smoke-run):')
print(res_one_shot)

# To verify, you can run `one_shot_ppm_update(res_one_shot['recommended_ppm'], target_batches=50)`
print("Whether one-shot recommended ppm meets target k_eff? \n K_eff at reommended ppm:", one_shot_ppm_update(res_one_shot['recommended_ppm'], target_batches=300)['k'])


/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


One-shot recommendation (quick smoke-run):
{'ppm': 1000.0, 'k': 1.0871937582926527, 'dk_dppm': -5.786288493861518e+20, 'recommended_ppm': 1000.0}


/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


Whether one-shot recommended ppm meets target k_eff? 
 K_eff at reommended ppm: 1.0871937582926545


## Clearly, one shot gradient approach does not give useful result since the dk_dppm is extremely large. Hence we iterate with a very small learning rate.

## Lets run the iterative gradient search version and compare it with the gradient-free (python api) k_eff search approach

In [13]:

ppm_start = 1000.0
k_target = 0.85
ppm_range = [0.0005, 20000]
max_iter = 50
compare_optimization_methods(ppm_start, k_target, ppm_range, max_iter)


COMPARING OPTIMIZATION METHODS FOR BORON CONCENTRATION SEARCH
Target k_eff: 0.85, Initial guess: 1000.0 ppm
GRADIENT-BASED OPTIMIZATION WITH ADAPTIVE LEARNING RATE
Initial: 1000.0 ppm, Target: k = 0.85
Iter |   ppm   |   k_eff   |  Error  |  Gradient  |  Step  | Learning Rate
-------------------------------------------------------------------------------------
NEW PPM SUGGESTION:  3103.9582831661355
ERROR WAS:  0.24154364950538965
  1 |  1000.0 |  1.091544 |  0.2415 |  -5.81e+20 |  2104.0 |     1.00e-17


/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


NEW PPM SUGGESTION:  3300.2395513561273
ERROR WAS:  0.06076566361745528
  2 |  3104.0 |  0.910766 |  0.0608 |  -3.23e+20 |   196.3 |     1.00e-17


/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


NEW PPM SUGGESTION:  3455.3264210489747
ERROR WAS:  0.05046490880278509
  3 |  3300.2 |  0.900465 |  0.0505 |  -3.07e+20 |   155.1 |     1.00e-17


...

/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


NEW PPM SUGGESTION:  3848.5174508409646
ERROR WAS:  0.014018291462160937
  9 |  3810.1 |  0.864018 |  0.0140 |  -2.74e+20 |    38.5 |     1.00e-17


/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/workspaces/openmc/openmc/mixin.py:70: IDWarning: Another UniverseBase instance already exists with id=0.
  warn(msg, IDWarning)


NEW PPM SUGGESTION:  3866.4255141926083
ERROR WAS:  0.009483110275060103
 10 |  3848.5 |  0.859483 |  0.0095 |  -2.70e+20 |    17.9 |     1.00e-17
✓ CONVERGED in 10 iterations


NameError: name 'builtin_ppm' is not defined

# Whether Tally derivatives can search boron concentration for an arbitrary target k_eff? (non-critical config. search)